
# Day 5 – Handling Missing Data, Encoding & Outlier Detection

## Learning Objectives
- Understand missing values and their impact
- Handle missing values using multiple strategies
- Perform Label Encoding and One-Hot Encoding
- Detect outliers using Quantile (IQR) Method
- Visualize outliers using Box Plots
- Apply complete preprocessing workflow on a real dataset

**Dataset Used:** Titanic Dataset (Seaborn)


In [ ]:

# Import Required Libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

print("Libraries Loaded Successfully")


## 1. Load Real Dataset

In [ ]:

# Load Titanic Dataset

df = sns.load_dataset('titanic')

print("Shape:", df.shape)
df.head()


In [ ]:

# Basic Information

df.info()


## 2. Understanding Missing Values

In [ ]:

# Count Missing Values

missing_values = df.isnull().sum()

missing_values.sort_values(ascending=False)


In [ ]:

# Visualize Missing Values

plt.figure(figsize=(10,5))
sns.heatmap(df.isnull(), cbar=False)

plt.title("Missing Values Heatmap")
plt.show()



## Strategy 1: Drop Missing Values

Use when:
- Very few rows contain missing values
- Missing values are random


In [ ]:

drop_df = df.dropna()

print("Original Shape:", df.shape)
print("After Drop:", drop_df.shape)


## Strategy 2: Fill Numerical Missing Values with Mean

In [ ]:

mean_df = df.copy()

mean_df['age'] = mean_df['age'].fillna(
    mean_df['age'].mean()
)

mean_df['age'].isnull().sum()


## Strategy 3: Fill Numerical Missing Values with Median

In [ ]:

median_df = df.copy()

median_df['age'] = median_df['age'].fillna(
    median_df['age'].median()
)

median_df['age'].isnull().sum()


## Strategy 4: Fill Categorical Missing Values with Mode

In [ ]:

mode_df = df.copy()

mode_df['embarked'] = mode_df['embarked'].fillna(
    mode_df['embarked'].mode()[0]
)

mode_df['embarked'].isnull().sum()


## 3. Understanding Encoding

In [ ]:

# Check Categorical Columns

df.select_dtypes(include='object').head()



### Label Encoding

Best for:
- Ordinal Data
- Ordered Categories


In [ ]:

label_df = df.copy()

encoder = LabelEncoder()

label_df['sex_encoded'] = encoder.fit_transform(
    label_df['sex']
)

label_df[['sex','sex_encoded']].head()



### One Hot Encoding

Best for:
- Nominal Data
- Unordered Categories


In [ ]:
# Create encoder
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown='ignore'
)

# Encode 'embarked' column
encoded_data = encoder.fit_transform(df[['embarked']])

# Create DataFrame from encoded values
encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(['embarked'])
)

# Combine with original DataFrame
final_df = pd.concat([df.drop('embarked', axis=1), encoded_df],
    axis=1
)

final_df.head()

## Label Encoding vs One-Hot Encoding

In [ ]:

comparison = pd.DataFrame({
    "Original": ["Male","Female","Male"],
    "Label Encoding": [1,0,1],
    "OneHot_Male":[1,0,1],
    "OneHot_Female":[0,1,0]
})

comparison


## 4. Understanding Outliers

In [ ]:

# Distribution of Fare

plt.figure(figsize=(8,5))

sns.histplot(df['fare'], bins=30)

plt.title("Fare Distribution")
plt.show()


## Quantile (IQR) Method

In [ ]:

Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)

IQR = Q3 - Q1

lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)


In [ ]:

# Detect Outliers

outliers = df[
    (df['fare'] < lower_limit) |
    (df['fare'] > upper_limit)
]

print("Number of Outliers:", len(outliers))

outliers.head()


## Box Plot Analysis

In [ ]:

plt.figure(figsize=(8,4))

sns.boxplot(x=df['fare'])

plt.title("Box Plot of Fare")
plt.show()


## Remove Outliers

In [ ]:

clean_df = df[
    (df['fare'] >= lower_limit) &
    (df['fare'] <= upper_limit)
]

print("Original Shape:", df.shape)
print("After Removing Outliers:", clean_df.shape)


## Complete Data Preprocessing Pipeline

In [ ]:

processed_df = df.copy()

# Fill Numerical Missing Values
processed_df['age'] = processed_df['age'].fillna(
    processed_df['age'].median()
)

# Fill Categorical Missing Values
processed_df['embarked'] = processed_df['embarked'].fillna(
    processed_df['embarked'].mode()[0]
)

# Label Encode Sex
processed_df['sex'] = LabelEncoder().fit_transform(
    processed_df['sex']
)

# Create encoder
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown='ignore'
)

# Fit and transform the embarked column
encoded_data = encoder.fit_transform(processed_df[['embarked']])

# Create DataFrame with encoded columns
encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(['embarked']),
    index=processed_df.index
)

# Drop original column and combine encoded columns
processed_df = pd.concat(
    [processed_df.drop('embarked', axis=1), encoded_df],
    axis=1
)

processed_df.head()



# Practice Questions

### Q1
How many missing values are present in each column?

### Q2
Replace missing age values using mean.

### Q3
Apply Label Encoding on sex column.

### Q4
Apply One-Hot Encoding on embarked column.

### Q5
Find outliers in fare using IQR method.

### Q6
Create a box plot for age and fare.

### Q7
Remove fare outliers and compare dataset shape.



# Key Takeaways

- Missing values reduce model quality.
- Mean, Median, Mode and Drop strategies are commonly used.
- Label Encoding is suitable for ordered categories.
- One-Hot Encoding is suitable for unordered categories.
- Quantile (IQR) method is a popular outlier detection technique.
- Box plots quickly identify outliers.
- Proper preprocessing improves model accuracy.
